# 04 - Dropout ablation (dropout ON vs dropout OFF)

**What this notebook does**: trains the same v3 MiniConvNet twice on the `faithful` split - once with
`dropout_rate = 0.5`, once with `0.0` - for the same epoch budget, and writes a clean 2-row table to
`outputs/ablation_dropout.csv`.

**What must already exist**: split CSVs from notebook 00. Run notebook 02 first so you already know
whether the architecture trains cleanly at all.

**Design decisions baked in**
* Identical budget for both arms - a truncated run makes dropout look worse than it is, because
  regularised models converge more slowly.
* Everything else (seed, learning rate, clipnorm, label smoothing, LR schedule, activation, split) is
  held identical, so the only difference between the two rows is dropout.
* Both arms are collapse-checked in both modes, and both save raw predictions (LESSONS 4, 5).

**What "looks right"**: two rows, both `status = ok`, both with `n_predicted_classes = 4`. A modest
difference either way is a normal outcome at this dataset size - what would *not* be normal is an arm
at ~0.25 accuracy (chance) or one with an all-zero confusion-matrix column.

**If an arm is invalid**: the comparison is meaningless and the delta must not be quoted. With all
five lesson-2 measures already active, that outcome is a finding to report, not a cue to start
tuning - see the README's "what improving accuracy means, and its limit".

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.config import *
from src.data_utils import resolve_data_root

ensure_dirs()
print('data root:', resolve_data_root())
print('ablation epoch budget:', EPOCHS_ABLATION)
print('dropout rate under test:', DROPOUT_RATE, 'vs 0.0')

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from src.data_utils import load_split, make_split_datasets, split_counts
from src.models import build_miniconvnet, count_params
from src.train_utils import (set_global_seeds, compute_report, compile_model, optimizer_summary,
                             class_weights_for, make_callbacks, make_epoch_timer, save_history,
                             plot_history, final_epoch_summary, run_name_for,
                             estimate_training_time)
from src.evaluate_utils import (predict, compute_metrics, detect_collapse, print_collapse_report,
                                plot_confusion_matrix, confusion, tumor_vs_subtype_breakdown,
                                save_predictions, record_ablation, load_results)

set_global_seeds(SEED)
for k, v in compute_report().items():
    print(f'{k}: {v}')

In [ ]:
SPLIT_FOR_ABLATION = 'faithful'   # keep faithful so the comparison is paper-comparable

sdf = load_split(SPLIT_FOR_ABLATION)
train_ds, val_ds, test_ds, frames = make_split_datasets(sdf, one_hot=True)
class_weight = class_weights_for(SPLIT_FOR_ABLATION, frames['train']['label'].values)

print(split_counts(sdf).to_string())
print('class_weight:', class_weight if class_weight else 'None (by design for faithful)')

## 1. Time estimate before either arm runs

In [ ]:
est = estimate_training_time(
    model_fn=lambda: compile_model(build_miniconvnet(), verbose=False),
    train_ds=train_ds, val_ds=val_ds,
    planned_epochs=EPOCHS_ABLATION, n_runs=2, class_weight=class_weight)

## 2. Arm runner

One function, used twice, with `dropout_rate` the only difference. The seed is re-set inside it so
both arms start from the same initialisation stream.

In [ ]:
def run_arm(dropout_rate, tag):
    set_global_seeds(SEED)
    run_name = run_name_for('ablation', SPLIT_FOR_ABLATION, tag)
    print('=' * 72)

    model = build_miniconvnet(dropout_rate=dropout_rate)
    compile_model(model)
    n_dropout = sum(1 for l in model.layers if l.__class__.__name__ == 'Dropout')
    print('run         :', run_name)
    print('dropout_rate:', dropout_rate, '| Dropout layers in model:', n_dropout)
    print('params      :', count_params(model)['total_params'])
    print('optimizer   :', optimizer_summary(model))

    timer = make_epoch_timer(verbose=0)
    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_ABLATION,
                        class_weight=class_weight,
                        callbacks=make_callbacks(run_name, timer=timer), verbose=2)
    save_history(history, run_name, timer=timer)
    summary = final_epoch_summary(history, timer=timer)
    print('\n', summary)

    y_true, y_pred, y_prob = predict(model, test_ds)
    metrics = compute_metrics(y_true, y_pred, y_prob)
    save_predictions(run_name, y_true, y_pred, y_prob,
                     meta={'split_variant': SPLIT_FOR_ABLATION, 'dropout_rate': dropout_rate,
                           'activation': ACTIVATION, 'label_smoothing': LABEL_SMOOTHING})

    collapse = detect_collapse(history=history, kappa=metrics['cohen_kappa'],
                               mcc=metrics['mcc'], y_pred=y_pred)
    breakdown = tumor_vs_subtype_breakdown(y_true, y_pred)

    print('\ntest metrics:', {k: round(v, 4) for k, v in metrics.items()})
    print('\nconfusion matrix:')
    print(confusion(y_true, y_pred))
    print()
    print_collapse_report(collapse, run_name)

    plot_history(history, run_name)
    plot_confusion_matrix(y_true, y_pred, run_name)

    record_ablation({
        'run_name': run_name,
        'split_variant': SPLIT_FOR_ABLATION,
        'dropout_enabled': bool(dropout_rate and dropout_rate > 0),
        'dropout_rate': dropout_rate,
        'accuracy': round(metrics['accuracy'], 6),
        'f1_macro': round(metrics['f1_macro'], 6),
        'cohen_kappa': round(metrics['cohen_kappa'], 6),
        'mcc': round(metrics['mcc'], 6),
        'n_predicted_classes': collapse['details'].get('n_predicted_classes'),
        'epochs_trained': summary['epochs_trained'],
        'status': collapse['status'],
        'notes': (f'budget {EPOCHS_ABLATION} epochs, identical seed/lr/clipnorm/label-smoothing '
                  f'across arms; activation={ACTIVATION}; '
                  f"{summary.get('total_minutes')} min on CPU"),
    })
    return {'run_name': run_name, 'metrics': metrics, 'collapse': collapse, 'history': history,
            'summary': summary, 'breakdown': breakdown, 'y_true': y_true, 'y_pred': y_pred}

## 3. Arm 1 - dropout ON (rate 0.5)

**Looks right**: `Dropout layers in model: 1`, and a train/val accuracy gap *smaller* than the
dropout-off arm.

In [ ]:
arm_on = run_arm(DROPOUT_RATE, 'dropout_on')

## 4. Arm 2 - dropout OFF (rate 0.0)

**Looks right**: `Dropout layers in model: 0`. If it prints 1, the rate was not threaded through and
the ablation is meaningless.

In [ ]:
arm_off = run_arm(0.0, 'dropout_off')

## 5. Comparison

Report the test-accuracy delta **together with** the train-minus-val gap: at this dataset size
dropout's effect on overfitting is usually clearer than its effect on test accuracy.

In [ ]:
def gap_of(res):
    s = res['summary']
    return round(s.get('final_accuracy', float('nan')) - s.get('final_val_accuracy', float('nan')), 4)

compare = pd.DataFrame([
    {'arm': 'dropout ON (0.5)', **{k: round(v, 4) for k, v in arm_on['metrics'].items()},
     'train_minus_val_acc': gap_of(arm_on),
     'n_classes': arm_on['collapse']['details'].get('n_predicted_classes'),
     'status': arm_on['collapse']['status']},
    {'arm': 'dropout OFF (0.0)', **{k: round(v, 4) for k, v in arm_off['metrics'].items()},
     'train_minus_val_acc': gap_of(arm_off),
     'n_classes': arm_off['collapse']['details'].get('n_predicted_classes'),
     'status': arm_off['collapse']['status']},
])
print(compare[['arm', 'accuracy', 'f1_macro', 'cohen_kappa', 'mcc', 'train_minus_val_acc',
               'n_classes', 'status']].to_string(index=False))

delta = arm_on['metrics']['accuracy'] - arm_off['metrics']['accuracy']
print(f'\ntest accuracy delta (ON - OFF): {delta:+.4f}')
print(f'overfitting gap: ON={gap_of(arm_on)} vs OFF={gap_of(arm_off)} '
      '(dropout should reduce this if it is doing anything)')

invalid = [a for a in (arm_on, arm_off) if a['collapse']['collapsed']]
if invalid:
    print(f'\n!!! {len(invalid)} arm(s) invalid ('
          + ', '.join(a['collapse']['status'] for a in invalid)
          + ') - the comparison is INVALID and the delta above must not be quoted.')
else:
    print('\nBoth arms valid and predicting all four classes - the delta is reportable.')

In [ ]:
abl = load_results('ablation')
print('outputs/ablation_dropout.csv')
print(abl.to_string(index=False))
print('\nnext: 05_train_baselines.ipynb')